# Day 2-01｜YOLO26 Detection 推論
> Python 籃球運動資料分析課程  
> 本單元使用已訓練 YOLO detector 對實際籃球影片執行物件偵測，觀察 class、confidence 與 bbox 輸出。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 載入 Ultralytics YOLO detector 權重。
- 對參考影片 frame 執行 detection。
- 產生一段含偵測框的 preview 影片與 JSON 結果。


## 執行階段提醒
請優先使用 **GPU** 或 **TPU** 的執行階段；不要使用純 CPU 執行。  
YOLO、MediaPipe 與影片處理在純 CPU 上會明顯較慢，容易讓課堂操作卡住。


## 課程流程
1. 選擇 `assets/raw/reference_videos/` 內的參考影片。
2. 對單一 frame 執行 YOLO detection。
3. 對短片段輸出 detection preview video。


In [ ]:
from pathlib import Path
import sys

bootstrap_candidates = [
    Path.cwd().resolve(),
    *Path.cwd().resolve().parents,
    Path("/content/basketball_hackathon/course"),
    Path("/content/basketball_hackathon_course"),
    Path("/content/drive/MyDrive/basketball_hackathon/course"),
]
for candidate in bootstrap_candidates:
    if (candidate / "src" / "course_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError(
        "找不到 src/course_setup.py。請先執行 init_colab.ipynb，確認課程已同步到 /content/basketball_hackathon/course 或目前 repo 內。"
    )

from src.course_setup import DEFAULT_COURSE_ROOT, bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(DEFAULT_COURSE_ROOT, mount_drive=True)


In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO

from src.cv_utils import save_image_rgb, show_image
from src.video_utils import display_video_in_notebook
from src.yolo_utils import detections_from_result, write_detection_preview_video

video_candidates = sorted((COURSE_ROOT / "assets" / "raw" / "reference_videos").glob("*.mp4"))
if not video_candidates:
    raise FileNotFoundError("找不到參考影片。請確認 assets/raw/reference_videos/ 內至少有一支 mp4。")

VIDEO_PATH = video_candidates[0]
MODEL_PATH = COURSE_ROOT / "assets" / "models" / "detectors" / "yolo26n_basketball_player_best.pt"
FRAME_INDEX = 15

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise FileNotFoundError(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_INDEX)
ok, frame_bgr = cap.read()
cap.release()
if not ok:
    raise RuntimeError(f"無法讀取 frame {FRAME_INDEX}: {VIDEO_PATH}")
frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

model = YOLO(str(MODEL_PATH))
result = model.predict(frame, conf=0.25, imgsz=960, verbose=False)[0]
detections = detections_from_result(result, frame_index=FRAME_INDEX)
vis = result.plot()[:, :, ::-1].copy()
show_image(vis, "YOLO26 detection result")

out_image = COURSE_ROOT / "assets" / "results" / "d2_01_yolo26_detection_frame.png"
save_image_rgb(out_image, vis)

print("video:", VIDEO_PATH)
print("model:", MODEL_PATH)
print("detections:", len(detections))
print("saved:", out_image)
pd.DataFrame([record.__dict__ for record in detections]).head()


In [ ]:
preview_path = COURSE_ROOT / "assets" / "results" / "d2_01_yolo26_detection_preview.mp4"
preview_path, preview_records = write_detection_preview_video(
    video_path=VIDEO_PATH,
    model_path=MODEL_PATH,
    output_path=preview_path,
    max_frames=120,
    conf=0.25,
    imgsz=960,
)
preview_json = preview_path.with_suffix(".json")

print("preview video:", preview_path)
print("preview json:", preview_json)
print("records:", len(preview_records))
display_video_in_notebook(preview_path, loop=True)


觀察重點：`confidence` 低的框不一定錯，但可能增加後續 tracking 的誤配；球與號碼尺寸小，通常比 player 類別更容易漏偵。

## 本單元產出檔案

- `assets/results/d2_01_yolo26_detection_frame.png`：單張 frame 的 detection 結果。
- `assets/results/d2_01_yolo26_detection_preview.mp4`：短片段 detection 預覽影片。
- `assets/results/d2_01_yolo26_detection_preview.json`：預覽影片逐 frame detection 摘要。
